In [1]:
import os
import sys
import numpy as np
import pandas as pd

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas11_KopiNusantara_MLlib") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
print(f"SparkSession aktif (versi {spark.version}). Total baris: {df.count()}")
df.printSchema()
df.show(5)

26/11/18 20:02:20 WARN Utils: Your hostname, revan18-1701 resolves to a loopback address: 127.0.1.1; using 192.168.18.236 instead (on interface wlp2s0)
26/11/18 20:02:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 20:02:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession aktif (versi 3.5.9). Total baris: 2500
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only sho

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Fit StringIndexer dan simpan modelnya agar bisa digunakan ulang pada Bagian E
model_idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
model_idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
model_idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

df_indexed = model_idx_menu.transform(df)
df_indexed = model_idx_cabang.transform(df_indexed)
df_indexed = model_idx_waktu.transform(df_indexed)

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_final = assembler.transform(df_indexed)

df_final.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_pesan_idx", "features", "total_penjualan").show(5, truncate=False)

+--------------+-------------+----------+----------+-----------+---------------+-------------------------+------------------+
|nama_menu     |nama_menu_idx|cabang    |cabang_idx|waktu_pesan|waktu_pesan_idx|features                 |total_penjualan   |
+--------------+-------------+----------+----------+-----------+---------------+-------------------------+------------------+
|Kentang Goreng|5.0          |Semarang  |2.0       |Malam      |1.0            |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Sore       |3.0            |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|5.0          |Semarang  |2.0       |Siang      |2.0            |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Pagi       |0.0            |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|Roti Bakar    |4.0          |Magelang  |0.0       |Malam      |1.0            |[2.0,30000.0,4.0,0.0,1.0]|58157.944120

In [4]:
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr.fit(train)

print("Model utama berhasil dilatih!")
print("Koefisien:", model_utama.coefficients)
print("Intercept:", model_utama.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661


Model utama berhasil dilatih!
Koefisien: [22046.663271025755,3.0229208681606696,-123.21425598657828,279.2845246167609,339.63879532317213]
Intercept: -66903.1776648588


In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model_utama.transform(test)
predictions.select("nama_menu", "cabang", "qty", "harga_satuan", "total_penjualan", "prediction").show(5)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = evaluator_rmse.evaluate(predictions)
r2_utama = evaluator_r2.evaluate(predictions)

print(f"RMSE Model Utama : Rp {rmse_utama:,.0f}")
print(f"R² Model Utama   : {r2_utama:.4f} ({r2_utama*100:.2f}% variasi data dijelaskan model)")

+---------+--------+---+------------+------------------+------------------+
|nama_menu|  cabang|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+---+------------+------------------+------------------+
|Americano|Magelang|  1|       25000|19721.170446051627|31056.146105506865|
|Americano|Magelang|  1|       25000|  21236.0956379149|31056.146105506865|
|Americano|Magelang|  4|       22000|  85834.8564401191| 88127.37331410211|
|Americano|Magelang|  4|       22000|102174.09100317833| 88127.37331410211|
|Americano|Magelang|  4|       30000|113044.63431555482|112310.74025938749|
+---------+--------+---+------------+------------------+------------------+
only showing top 5 rows



RMSE Model Utama : Rp 10,901
R² Model Utama   : 0.9107 (91.07% variasi data dijelaskan model)


**Interpretasi:**  
Model regresi linear utama menghasilkan R² sebesar 0,9107 (91,07% variasi `total_penjualan` berhasil dijelaskan oleh model) dengan RMSE sebesar Rp 10.901. Nilai R² di atas 0,90 ini menunjukkan model sudah sangat baik dan layak digunakan sebagai alat bantu estimasi omzet pesanan Kopi Nusantara, meskipun masih ada sedikit galat karena hubungan perkalian `qty * harga_satuan` bersifat non-linear bagi regresi linear murni.

In [6]:
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)

train_sed = assembler_sederhana.transform(train)
test_sed = assembler_sederhana.transform(test)

lr_sed = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model_sederhana = lr_sed.fit(train_sed)

pred_sed = model_sederhana.transform(test_sed)
rmse_sed = evaluator_rmse.evaluate(pred_sed)
r2_sed = evaluator_r2.evaluate(pred_sed)

print(f"Model Utama (5 fitur)     -> RMSE: Rp {rmse_utama:,.0f} | R²: {r2_utama:.4f}")
print(f"Model Sederhana (2 fitur) -> RMSE: Rp {rmse_sed:,.0f} | R²: {r2_sed:.4f}")
print(f"Selisih R² (Utama - Sederhana): {r2_utama - r2_sed:+.6f}")

Model Utama (5 fitur)     -> RMSE: Rp 10,901 | R²: 0.9107
Model Sederhana (2 fitur) -> RMSE: Rp 10,896 | R²: 0.9108
Selisih R² (Utama - Sederhana): -0.000078


**Kesimpulan Perbandingan:**  
Nilai R² model kedua (2 fitur: `qty` dan `harga_satuan`) hampir identik dengan model utama (5 fitur), perbedaannya sangat kecil (di bawah 0,001). Hal ini berarti menambahkan informasi `nama_menu`, `cabang`, dan `waktu_pesan` **tidak meningkatkan akurasi prediksi secara berarti** pada dataset ini, karena secara matematis `total_penjualan` memang ditentukan langsung oleh `qty * harga_satuan` tanpa pengaruh signifikan dari menu, cabang, maupun waktu pesan.

In [7]:
skenario_baru = spark.createDataFrame([
    ("Kopi Susu", "Magelang", "Pagi", 3, 20000),
    ("Cappuccino", "Yogyakarta", "Sore", 2, 25000),
    ("Roti Bakar", "Semarang", "Malam", 5, 18000),
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# Gunakan model StringIndexer yang SAMA dari Bagian A (tanpa fit ulang)
skenario_idx = model_idx_menu.transform(skenario_baru)
skenario_idx = model_idx_cabang.transform(skenario_idx)
skenario_idx = model_idx_waktu.transform(skenario_idx)

skenario_final = assembler.transform(skenario_idx)
hasil_prediksi = model_utama.transform(skenario_final)

hasil_prediksi.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |3  |20000       |59572.01525544528|
|Cappuccino|Yogyakarta|Sore       |2  |25000       |53814.94297982256|
|Roti Bakar|Semarang  |Malam      |5  |18000       |98148.0651377724 |
+----------+----------+-----------+---+------------+-----------------+



In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
